In [1]:
import pandas as pd
import numpy as np

DATA_PATH = '../data/ml-100k/'

ratings = pd.read_csv(DATA_PATH + 'u.data',sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])
ratings_sorted = ratings.sort_values(by='timestamp')

split_index = int(0.8 * len(ratings_sorted))
train = ratings_sorted.iloc[:split_index]
test = ratings_sorted.iloc[split_index:]

print(f'Train: {len(train)} оценок')
print(f'Test:  {len(test)} оценок')
print(f'Train период: {train["timestamp"].min()} — {train["timestamp"].max()}')
print(f'Test период:  {test["timestamp"].min()} — {test["timestamp"].max()}')

Train: 80000 оценок
Test:  20000 оценок
Train период: 874724710 — 889237269
Test период:  889237269 — 893286638


In [2]:
ratings.head(5)

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [3]:
from sklearn.metrics import root_mean_squared_error

global_mean = train['rating'].mean()
test_pred_global = np.full(len(test), global_mean)

rmse_global = root_mean_squared_error(test['rating'], test_pred_global)

print(f'Global mean: {global_mean:.3f}')
print(f'RMSE (global mean baseline): {rmse_global:.4f}')

Global mean: 3.518
RMSE (global mean baseline): 1.1191


In [4]:
user_means = train.groupby('user_id')['rating'].mean()
test_pred_user = test['user_id'].map(user_means).fillna(global_mean)
rmse_user =  root_mean_squared_error(test['rating'], test_pred_user)

item_means = train.groupby('item_id')['rating'].mean()
test_pred_item = test['item_id'].map(item_means).fillna(global_mean)
rmse_item = root_mean_squared_error(test['rating'], test_pred_item)

print(f'RMSE (global mean): {rmse_global:.4f}')
print(f'RMSE (user means baseline): {rmse_user:.4f}')
print(f'RMSE (item means baseline): {rmse_item:.4f}')

RMSE (global mean): 1.1191
RMSE (user means baseline): 1.1258
RMSE (item means baseline): 1.0367


**global_mean** бьёт **user_mean** потому что датасет разбит по времени и при предсказании многие юзеры становятся новыми по этому в user_mean алгоритме мы заполняем их значениями global_mean, + добавляется шум из-за того что некоторых мы делим так что в трейне остаётся 1-2 его оценки и средние по такой выборке делает только хуже. Проверим сколько юзеров появляется только в тесте. 

In [5]:
users_in_train = set(train['user_id'].unique())
users_in_test = set(test['user_id'].unique())
new_users = users_in_test - users_in_train

print(f'Юзеров в test: {len(users_in_test)}')
print(f'Из них новых (нет в train): {len(new_users)}')

# сколько оценок в test принадлежат новым юзерам
n_new_ratings = test['user_id'].isin(new_users).sum()
print(f'Оценок в test от новых юзеров: {n_new_ratings} из {len(test)}')

Юзеров в test: 301
Из них новых (нет в train): 192
Оценок в test от новых юзеров: 17048 из 20000


Проанализировав split, я получил, что 85% тестовых оценок принадлежат юзерам, отсутствующим в train. Это заранее объясняет, почему user-based подходы здесь проседают, а item-based и content-based устойчивее.

In [11]:
train.to_csv('../data/processed_train.csv', index=False)
test.to_csv('../data/processed_test.csv', index=False)
print('train/test сохранены')

train/test сохранены
